## Import

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"   # single-threaded so parallel jobs do not fight over cores

import json, os, time, pickle, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

/Users/waroonragwongsiri/Playground/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
SEED = 42
N_TRIALS = 30   # same Optuna budget for every model
INNER_K = 3
K, REPEATS = 5, 3
os.makedirs("weights_V6_1", exist_ok=True)

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()  # 2213 rows (target must not be imputed)
print("cleaned:", df.shape, "| Income max:", f"{df['Income'].max():,.0f}", "| median:", f"{df['Income'].median():,.0f}")
REG_FEATURES = ["NumCatalogPurchases","MntMeatProducts","MntWines","NumWebVisitsMonth","MntSweetProducts",
    "MntFruits","NumWebPurchases","MntGoldProds","NumStorePurchases","MntFishProducts","Kidhome","Age","NumDealsPurchases"]
X_full = df[REG_FEATURES].values; y_raw = df["Income"].values.astype(float)
print(f"features: {len(REG_FEATURES)} dims (all numeric)")

cleaned: (2212, 28) | Income max: 162,397 | median: 51,371
features: 13 dims (all numeric)


In [4]:
def prep_split(tr_idx, te_idx):
    """Standardise using the training rows only (no missing values left in the regression set)."""
    sc = StandardScaler().fit(X_full[tr_idx])
    return sc.transform(X_full[tr_idx]), sc.transform(X_full[te_idx]), y_raw[tr_idx], y_raw[te_idx]

## Model

In [5]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

def _scaled_y(est): return TransformedTargetRegressor(regressor=est, transformer=StandardScaler())

class OneFeature(FunctionTransformer):
    def __init__(self, j=0): super().__init__(func=None); self.j = j
    def fit(self, X, y=None): return self
    def transform(self, X): return X[:, [self.j]]

SPECS = {
 "Linear regression (1 feature)": dict(tier="curriculum",
    space=lambda t: dict(j=t.suggest_int("j", 0, len(REG_FEATURES) - 1)),
    make=lambda p, s: make_pipeline(OneFeature(p["j"]), LinearRegression())),
 "Multiple regression": dict(tier="curriculum", fixed=True, space=lambda t: {},
    make=lambda p, s: LinearRegression()),
 "Polynomial regression": dict(tier="curriculum",
    space=lambda t: dict(deg=t.suggest_int("deg", 2, 3), a=t.suggest_float("a", 1e-2, 1e4, log=True)),
    make=lambda p, s: make_pipeline(PolynomialFeatures(p["deg"], include_bias=False), StandardScaler(), Ridge(alpha=p["a"]))),
 "Decision tree": dict(tier="curriculum family",
    space=lambda t: dict(d=t.suggest_int("d", 2, 12), leaf=t.suggest_int("leaf", 1, 30)),
    make=lambda p, s: DecisionTreeRegressor(max_depth=p["d"], min_samples_leaf=p["leaf"], random_state=s)),
 "Random forest": dict(tier="curriculum family",
    space=lambda t: dict(n=t.suggest_int("n", 100, 400, step=100), d=t.suggest_int("d", 3, 16),
                         leaf=t.suggest_int("leaf", 1, 10), mf=t.suggest_categorical("mf", [1.0, 0.5, 0.33])),
    make=lambda p, s: RandomForestRegressor(n_estimators=p["n"], max_depth=p["d"], min_samples_leaf=p["leaf"],
                         max_features=p["mf"], n_jobs=1, random_state=s)),
 "ExtraTrees": dict(tier="curriculum family",
    space=lambda t: dict(n=t.suggest_int("n", 100, 400, step=100), d=t.suggest_int("d", 3, 16),
                         leaf=t.suggest_int("leaf", 1, 10), mf=t.suggest_categorical("mf", [1.0, 0.5, 0.33])),
    make=lambda p, s: ExtraTreesRegressor(n_estimators=p["n"], max_depth=p["d"], min_samples_leaf=p["leaf"],
                         max_features=p["mf"], n_jobs=1, random_state=s)),
 "Gradient boosting": dict(tier="curriculum family",
    space=lambda t: dict(n=t.suggest_int("n", 50, 300, step=50), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                         d=t.suggest_int("d", 2, 5), sub=t.suggest_float("sub", 0.6, 1.0)),
    make=lambda p, s: GradientBoostingRegressor(n_estimators=p["n"], learning_rate=p["lr"], max_depth=p["d"],
                         subsample=p["sub"], random_state=s)),
 "HistGradientBoosting": dict(tier="curriculum family",
    space=lambda t: dict(it=t.suggest_int("it", 50, 400, step=50), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                         leaf=t.suggest_int("leaf", 8, 64), l2=t.suggest_float("l2", 1e-3, 10, log=True)),
    make=lambda p, s: HistGradientBoostingRegressor(max_iter=p["it"], learning_rate=p["lr"], max_leaf_nodes=p["leaf"],
                         l2_regularization=p["l2"], random_state=s)),
 "kNN": dict(tier="curriculum family",
    space=lambda t: dict(k=t.suggest_int("k", 3, 51, step=2), w=t.suggest_categorical("w", ["uniform", "distance"]),
                         p=t.suggest_int("p", 1, 2)),
    make=lambda p, s: KNeighborsRegressor(n_neighbors=p["k"], weights=p["w"], p=p["p"])),
 "SVR (RBF)": dict(tier="curriculum family",
    space=lambda t: dict(C=t.suggest_float("C", 1e-1, 1e2, log=True), g=t.suggest_float("g", 1e-4, 1.0, log=True),
                         e=t.suggest_float("e", 0.01, 0.5, log=True)),
    make=lambda p, s: _scaled_y(SVR(C=p["C"], gamma=p["g"], epsilon=p["e"]))),
 "MLP (sklearn)": dict(tier="curriculum family",
    space=lambda t: dict(h1=t.suggest_int("h1", 16, 128, step=16), two=t.suggest_categorical("two", [False, True]),
                         a=t.suggest_float("a", 1e-5, 1e-1, log=True), lr=t.suggest_float("lr", 1e-3, 1e-2, log=True)),
    make=lambda p, s: _scaled_y(MLPRegressor(hidden_layer_sizes=(p["h1"], p["h1"] // 2) if p["two"] else (p["h1"],),
                         alpha=p["a"], learning_rate_init=p["lr"], max_iter=400, early_stopping=True,
                         n_iter_no_change=15, random_state=s))),
}

## Sanity Check

In [6]:
from sklearn.dummy import DummyRegressor
a, b = next(KFold(K, shuffle=True, random_state=SEED).split(X_full))
assert len(set(a) & set(b)) == 0, "train / eval rows overlap"
Xa, Xb, ya, yb = prep_split(a, b)
pred = DummyRegressor().fit(Xa, ya).predict(Xb)
print(f"mean-only baseline: R2 {r2_score(yb, pred):.4f} | RMSE {np.sqrt(mean_squared_error(yb, pred)):,.0f}")
assert abs(r2_score(yb, pred)) < 0.02

mean-only baseline: R2 -0.0038 | RMSE 21,836


## Training

In [7]:
def inner_oof(spec, params, X, y, seed):
    """Out-of-fold predictions on the training rows (inner K-fold)."""
    oof = np.zeros(len(y))
    for a, b in KFold(INNER_K, shuffle=True, random_state=seed).split(X):
        est = spec["make"](params, seed)
        est.fit(X[a], y[a])
        oof[b] = est.predict(X[b])
    return oof

def tune(spec, X, y, seed):
    """Optuna (TPE) minimising RMSE of the inner out-of-fold predictions."""
    if spec.get("fixed"):
        return {}
    def objective(trial):
        return float(np.sqrt(mean_squared_error(y, inner_oof(spec, spec["space"](trial), X, y, seed))))
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=N_TRIALS)
    return spec["space"](optuna.trial.FixedTrial(study.best_params))

def fit_eval(spec, params, Xa, ya, Xb, yb, seed):
    est = spec["make"](params, seed)
    est.fit(Xa, ya)
    pred = est.predict(Xb)
    mse = mean_squared_error(yb, pred)
    return {"r2": r2_score(yb, pred), "rmse": float(np.sqrt(mse)), "mae": mean_absolute_error(yb, pred), "mse": mse}, pred

In [8]:
cache = "weights_V6_1/_progress_regression_sklearn.pkl"
state = pickle.load(open(cache, "rb")) if os.path.exists(cache) else {"rows": [], "scores": {}}
done = {(r["model"], r["repeat"], r["fold"]) for r in state["rows"]}
for rep in range(1, REPEATS + 1):
    seed_r = SEED + rep - 1               # repeat 1 uses the same folds as V1_3 / V3_1
    for fold, (a, b) in enumerate(KFold(K, shuffle=True, random_state=seed_r).split(X_full), start=1):
        Xa, Xb, ya, yb = prep_split(a, b)
        for name, spec in SPECS.items():
            if (name, rep, fold) in done:
                continue
            t0 = time.time()
            params = tune(spec, Xa, ya, seed_r * 100 + fold)
            row, s = fit_eval(spec, params, Xa, ya, Xb, yb, seed_r * 100 + fold)
            state["rows"].append(dict(model=name, tier=spec["tier"], selection="inner validation", repeat=rep, fold=fold,
                                      n_test=int(len(yb)), **{k: round(float(v), 4) for k, v in row.items()},
                                      params=params, seconds=round(time.time() - t0, 1)))
            key = f"{name}|r{rep}"
            state["scores"].setdefault(key, np.full(len(y_all if "reg" == "clf" else y_raw), np.nan))[b] = s
        pickle.dump(state, open(cache, "wb"))
        print(f"repeat {rep} fold {fold} done | rows so far {len(state['rows'])}", flush=True)

repeat 1 fold 1 done | rows so far 11


repeat 1 fold 2 done | rows so far 22


repeat 1 fold 3 done | rows so far 33


repeat 1 fold 4 done | rows so far 44


repeat 1 fold 5 done | rows so far 55


...

repeat 3 fold 1 done | rows so far 121


repeat 3 fold 2 done | rows so far 132


repeat 3 fold 3 done | rows so far 143


repeat 3 fold 4 done | rows so far 154


repeat 3 fold 5 done | rows so far 165


## K-Fold Summary

In [9]:
kdf = pd.DataFrame(state["rows"])
KEYS = ['r2', 'rmse', 'mae']
summary = {m: {k: {"mean": round(float(g[k].mean()), 4), "std": round(float(g[k].std()), 4)} for k in KEYS if k in g and g[k].notna().any()}
           for m, g in kdf.groupby("model")}
rank = kdf.groupby("model")["r2"].agg(["mean", "std"]).sort_values("mean", ascending=False).round(4)
print(rank.to_string())
json.dump({"k": K, "repeats": REPEATS, "n_trials": N_TRIALS, "folds": state["rows"], "summary": summary},
          open("weights_V6_1/regression_sklearn_kfold_metrics.json", "w"), indent=2, default=str)
np.savez("weights_V6_1/regression_sklearn_kfold_scores.npz", **state["scores"])

                                 mean     std
model                                        
ExtraTrees                     0.8208  0.0692
Gradient boosting              0.8168  0.0693
HistGradientBoosting           0.8133  0.0633
Random forest                  0.7969  0.0588
SVR (RBF)                      0.7889  0.0600
MLP (sklearn)                  0.7692  0.0921
kNN                            0.7686  0.0592
Polynomial regression          0.7511  0.0692
Decision tree                  0.7383  0.0781
Multiple regression            0.7332  0.0431
Linear regression (1 feature)  0.4463  0.0643
